# FINS notebook 3 — learned operators and published baselines on the new equations (Colab, GPU)

Everything here is registered in `README_EXPERIMENTS.md` (07.10) before any of it runs. In order of priority:
* cell 3: FNO steppers (exp39's recipe: 1024 trajectories, 400 epochs, 64 modes) — Benjamin–Ono seeds 43, 44;
  Kuramoto–Sivashinsky seeds 42–44 on the corrected `mixed` data (attractor states, half on a constant background,
  as KdV's and BO's) and, as an ablation, on zero-mean `attractor` data. Steppers that exist are reused: notebook 2's
  KS steppers were trained on zero-mean data and become the ablation (cell 2). Each is checked by pointwise FINS on
  data seed 1007.
* cell 4: FINS through the learned operators — KdV weak form at noise 1e-3 and 1e-2 (the 6 runs of exp51 left),
  BO (6 runs), KS (12 runs).
* cell 5: the 50 KS weak-form runs of notebook 2 (skipped where done).
* cell 6: LieGAN, LaLiGAN, Augerino on BO and KS, seeds 42–44 (~30 min per run).
* cell 7: LIG on KS (protocol settings: 10 epochs, batch 1), its untrained null first; ~2 h per seed. If the GPU
  budget runs out, stop after cell 6 and send what exists.
* cell 8 shows progress; cell 9 zips the outputs and the new checkpoints for download.

Before running, upload to My Drive (root or one folder down) the newest **`fins_repo_full.zip`** and
**`fins_ckpt_nb3.zip`** from the chat. Outputs go to `MyDrive/fins_runs/results`, as in notebooks 1–2. After a
disconnect rerun cell 1 and the interrupted cell: finished runs are skipped, a stepper training resumes from its
partial state, a baseline run resumes at its first missing method.

In [ ]:
# 1. setup: Drive, the newest repository zip with notebook 3's code, the checkpoints, dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(R, exist_ok=True)
MARK = 'ks-mixed-background-v2'          # in exp39 since notebook 3


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_nb3(z):
    with zipfile.ZipFile(z) as zz:
        e39 = 'fins_repo/experiments/exp39_localised.py'
        return e39 in zz.namelist() and MARK in zz.read(e39).decode()


ZIP_PATH = newest('fins_repo_full*.zip', has_nb3)
if ZIP_PATH is None:
    raise SystemExit("no fins_repo_full*.zip on Drive has notebook 3's code: upload the newest zip from the chat")
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
CK = newest('fins_ckpt_nb3*.zip', lambda z: True)
if CK is None:
    raise SystemExit('fins_ckpt_nb3.zip is not on Drive: upload it from the chat')
with zipfile.ZipFile(CK) as z:
    for n in z.namelist():
        dst = os.path.join(R, os.path.basename(n))
        if n.endswith('.pt') and not os.path.exists(dst):
            with z.open(n) as src, open(dst, 'wb') as out:
                shutil.copyfileobj(src, out)
            print('checkpoint', os.path.basename(n))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'torchdiffeq'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')


def stream(cmd):
    t0 = time.time()
    with subprocess.Popen(cmd, cwd=EXPDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
    print(f'exit {p.returncode} after {time.time() - t0:.0f} s')

In [ ]:
# 2. notebook 2 trained the KS steppers on zero-mean data under the name 'mixed': they become the 'attractor' ablation,
#    and 'mixed' is retrained on data with constant backgrounds (README, 07.10). Runs once (marker file on Drive).
import torch
marker = os.path.join(R, 'ks_steppers_renamed_nb3.txt')
if os.path.exists(marker):
    print('already done:', open(marker).read())
else:
    log = []
    for s in (42, 43, 44):
        old = os.path.join(R, f'S_theta_ks_s{s}_mixed_stepper.pt')
        new = os.path.join(R, f'S_theta_ks_s{s}_attractor_stepper.pt')
        if os.path.exists(old + '.partial'):
            os.remove(old + '.partial')            # a zero-mean training state: never resume from it
            log.append(f'removed {os.path.basename(old)}.partial')
        if os.path.exists(old) and torch.load(old, map_location='cpu', weights_only=False).get('recipe') is None:
            if os.path.exists(new):
                raise SystemExit(f'{old} (zero mean) and {new} both exist: check by hand')
            os.rename(old, new)
            log.append(f'{os.path.basename(old)} -> {os.path.basename(new)}')
            for ext in ('.npz', '.log'):
                chk = os.path.join(R, 'steppers', f'pointwise_ks_stepper_s{s}{ext}')
                if os.path.exists(chk):
                    os.rename(chk, os.path.join(R, 'steppers', f'pointwise_ks_attractor_stepper_s{s}{ext}'))
                    log.append(f'{os.path.basename(chk)} -> pointwise_ks_attractor_stepper_s{s}{ext}')
    open(marker, 'w').write('\n'.join(log) or 'nothing to rename')
    print('\n'.join(log) or 'nothing to rename')

In [ ]:
# 3. the FNO steppers that are missing, each checked by pointwise FINS on data seed 1007 (~10-20 min per training)
os.makedirs(os.path.join(R, 'steppers'), exist_ok=True)
TODO = [('bo', 'mixed', 43), ('bo', 'mixed', 44)] + [('ks', d, s) for d in ('mixed', 'attractor') for s in (42, 43, 44)]
for pde, data, s in TODO:
    out = os.path.join(R, 'steppers', f'pointwise_{pde}_{data}_stepper_s{s}.npz')
    if os.path.exists(out):
        print('done', os.path.basename(out))
        continue
    log = out.replace('.npz', '.log')
    cmd = [sys.executable, '-u', 'exp39_localised.py', '--pde', pde, '--op', 'stepper', '--train-ic', data,
           '--train-if-missing', '--seed', str(s), '--data-seed', '1007', '--ic', 'periodic', '--n', '10', '--K', '16',
           '--every', '16', '--interior', '--ut', 'fd', '--save', out]
    t0 = time.time()
    with open(log, 'a') as fh:
        rc = subprocess.call(cmd, cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    keep = [l.strip() for l in open(log).read().splitlines() if 'one-step' in l or 'algebra inside' in l]
    print(pde, data, s, 'exit', rc, f'{time.time() - t0:.0f} s')
    for l in keep[-2:]:
        print('   ', l[:120])

In [ ]:
# 4. FINS through the learned operators (registered): KdV weak form at 1e-3 and 1e-2 (exp51), BO and KS (exp52, exp53)
stream([sys.executable, '-u', 'exp51_noise_weak.py', 'run', '--pdes', 'kdv', '--parts', 'weak-learned',
        '--conds', 'n1e-3,n1e-2', '--seeds', '42,43,44', '--runs', os.path.join(R, 'noise_weak_runs')])
stream([sys.executable, '-u', 'exp52_bo.py', 'run', '--parts', 'weak-learned,pointwise-learned', '--conds', 'clean',
        '--seeds', '42,43,44', '--runs', os.path.join(R, 'bo_runs')])
stream([sys.executable, '-u', 'exp53_ks.py', 'run', '--parts',
        'weak-learned,pointwise-learned,weak-learned-attractor,pointwise-learned-attractor', '--conds', 'clean',
        '--seeds', '42,43,44', '--runs', os.path.join(R, 'ks_runs')])

In [ ]:
# 5. the 50 registered KS weak-form runs of notebook 2 (skipped where done)
stream([sys.executable, '-u', 'exp53_ks.py', 'run', '--parts', 'weak', '--runs', os.path.join(R, 'ks_runs')])

In [ ]:
# 6. LieGAN, LaLiGAN, Augerino on BO and KS, seeds 42-44 (exp54 part 'gan'; ~30 min per run)
stream([sys.executable, '-u', 'exp54_baselines_new.py', 'run', '--parts', 'gan', '--root', R, '--threads', '2'])

In [ ]:
# 7. LIG on KS, protocol settings (10 epochs, batch 1, torchdiffeq adjoint): its untrained null, then seeds 42, 43, 44
#    (~2 h each). Optional: stop here if the GPU budget runs out; finished seeds are kept.
stream([sys.executable, '-u', 'exp54_baselines_new.py', 'run', '--parts', 'lig-null,lig', '--pdes', 'ks', '--root', R,
        '--threads', '2'])

In [ ]:
# 8. progress (safe while another cell runs or after it stopped)
def n(pat):
    return len(glob.glob(os.path.join(R, pat)))


print('steppers:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(R, 'S_theta_*_stepper*.pt'))
                          if '_bo_' in p or '_ks_' in p))
for f in sorted(glob.glob(os.path.join(R, 'steppers', '*.log'))):
    ep = [l for l in open(f).read().splitlines() if 'epoch' in l]
    print('  ', os.path.basename(f), '|', ep[-1].strip()[:100] if ep else '')
print('KdV weak, learned, 1e-3 / 1e-2:', n('noise_weak_runs/n1e-3/weak-learned_kdv_s*.npz'),
      n('noise_weak_runs/n1e-2/weak-learned_kdv_s*.npz'), 'of 3 each')
for pde, d in (('bo', 'bo_runs'), ('ks', 'ks_runs')):
    print(pde, 'clean:', {t: n(f'{d}/clean/{t}_{pde}_s*.npz') for t in
                          ('weak-learned', 'pointwise-learned', 'weak-learned-attractor', 'pointwise-learned-attractor',
                           'baselines', 'baselines-lig-null', 'baselines-lig')})
print('KS weak:', {os.path.basename(d): n(f'ks_runs/{os.path.basename(d)}/weak_ks_s*.npz')
                   for d in sorted(glob.glob(os.path.join(R, 'ks_runs', '*')))})
for f in sorted(glob.glob(os.path.join(R, '*_runs', 'clean', 'logs', 'baselines*.log'))):
    ll = [l for l in open(f).read().splitlines() if l.strip()]
    print('  ', os.path.basename(f), '|', ll[-1][:100] if ll else '')

In [ ]:
# 9. zip what exists (npz + logs of this notebook's runs, the stepper checks, the new checkpoints); download, a copy
#    stays on Drive
from google.colab import files
stage = '/content/fins_nb3'
shutil.rmtree(stage, ignore_errors=True)


def put(pattern):
    for p in glob.glob(os.path.join(R, pattern)):
        if p.endswith(('.claim', '.partial')) or os.path.isdir(p):
            continue
        dst = os.path.join(stage, os.path.relpath(p, R))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst)


for c in ('n1e-3', 'n1e-2'):
    put(f'noise_weak_runs/{c}/weak-learned_kdv_s*.npz')
    put(f'noise_weak_runs/{c}/logs/weak-learned_kdv_s*.log')
for d in ('bo_runs', 'ks_runs'):
    for pat in ('*learned*.npz', 'baselines*.npz', 'logs/*learned*.log', 'logs/baselines*.log'):
        put(f'{d}/clean/{pat}')
put('ks_runs/*/weak_ks_s*.npz')
put('ks_runs/*/logs/weak_ks_s*.log')
put('steppers/*')
put('ks_steppers_renamed_nb3.txt')
for p in ['S_theta_bo_s43_mixed_stepper.pt', 'S_theta_bo_s44_mixed_stepper.pt'] + \
        [f'S_theta_ks_s{s}_{d}_stepper.pt' for d in ('mixed', 'attractor') for s in (42, 43, 44)]:
    put(p)
arc = shutil.make_archive('/content/fins_nb3', 'zip', stage)
shutil.copy(arc, '/content/drive/MyDrive/fins_nb3.zip')
print(arc, os.path.getsize(arc), 'bytes; a copy is on Drive')
files.download(arc)